In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd().parents[0]
os.chdir(repo_root)

print(Path.cwd())

In [ ]:
repo_root

In [ ]:
from scripts.data_loader import load_locality_data
import scripts.weed_simulation_tools as wst

In [ ]:
localities = load_locality_data()

metro = localities[
    localities["postcode"].astype(int).between(6000,6199)
].copy()

metro['weed_index'] = wst.weed_index_initialize(density=metro['population_density_km2'],
                                            pressure_coef=0.05,
                                            w_min=0.05,
                                            w_max=1)

In [ ]:
type(localities)

In [ ]:
type(metro)

In [ ]:
metro.columns

In [ ]:
metro.shape

In [ ]:
metro["population_density_km2"].hist()

In [ ]:
metro["weed_index"].hist()

In [ ]:
metro["population_density_km2"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
)

In [ ]:
metro["weed_index"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
)

In [ ]:
for coefficient in [0.05, 0.005, 0.001, 0.000783, 0.0005]:
    column_name = f"weed_index_k_{coefficient}"

    metro[column_name] = wst.weed_index_initialize(
        density=metro["population_density_km2"],
        pressure_coef=coefficient,
        w_min=0.05,
        w_max=1.0,
    )

    print(f"\npressure_coef = {coefficient}")
    print(
        metro[column_name].describe(
            percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
        )
    )

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))

metro.plot(
    column="weed_index",
    cmap="YlGn",
    legend=True,
    ax=ax,
    edgecolor="none"
)

ax.set_title("Initial Weed Density Index")
ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
len(metro)

In [ ]:
import scripts.weed_simulation_tools as wst
import numpy as np

In [ ]:
weed_grid, valid_mask, transform = wst.polygon_to_grid(
    metro,
    cell_size=1000
)

print(weed_grid.shape)
print(np.nanmin(weed_grid))
print(np.nanmax(weed_grid))

In [ ]:
weed_grid

In [ ]:
import matplotlib.pyplot as plt

cmap = plt.cm.YlGn.copy()
cmap.set_bad("lightgray")

plt.figure(figsize=(10, 10))

plt.imshow(
    weed_grid,
    cmap=cmap,
    vmin=0,
    vmax=1
)

plt.colorbar(label="Weed Index")

plt.title("Initial Weed Index Grid")
plt.axis("off")

plt.show()

In [ ]:
import importlib
import scripts.weed_ca

importlib.reload(scripts.weed_ca)

weedca = scripts.weed_ca.WeedCA.polygon_to_weedca(metro, 
                                                  pressure_coef=0.000783,
                                                  cell_size=1000,
                                                  growth_rate=0.02,
                                                  diffusion_rate=0.01)

modes = ['growth', 'diffusion']
weedca.animate(frames=108, fps=20, modes=modes)

In [ ]:
def simple_removal(weed_grid, mask):
    weed_grid[~mask] = 0
    diff = weed_grid * 0.5
    cost = np.sum(diff * 200)

    return diff, cost

weedca.register_policy(simple_removal)

In [ ]:
before_grid = weedca.weed_grid.copy()
before_cost = weedca.total_cost

weedca.step(modes=["removal"])

after_grid = weedca.weed_grid
after_cost = weedca.total_cost

In [ ]:
print("Step:", weedca.step_counter)
print("Cost before:", before_cost)
print("Cost after:", after_cost)

print(
    "Invalid cells remain NaN:",
    np.all(np.isnan(after_grid[~weedca.valid_mask])),
)

print(
    "All valid cells decreased or stayed equal:",
    np.all(
        after_grid[weedca.valid_mask]
        <= before_grid[weedca.valid_mask]
    ),
)

print(
    "No valid cell became negative:",
    np.all(
        after_grid[weedca.valid_mask] >= 0
    ),
)

In [ ]:
modes = ['growth', 'diffusion', 'removal']
weedca.animate(frames=108, fps=20, modes=modes)

In [ ]:
history, cost_history = weedca.simulate(
    steps=108,
    modes=["growth", "diffusion", "removal"],
)

print("History shape:", history.shape)
print("Cost history shape:", cost_history.shape)

In [ ]:
mean_weed_history = np.nanmean(
    history,
    axis=(1, 2),
)

plt.figure(figsize=(10, 5))

plt.plot(
    mean_weed_history,
    label=f"Removal rate = {0.5:.1%}",
)

plt.xlabel("Simulation Step")
plt.ylabel("Mean Weed Index")
plt.title("Mean Weed Index Over Time")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
removal_rates = [
    0.0,
    0.001,
    0.01,
    0.1,
]

results = {}

for removal_rate in removal_rates:

    weedca = scripts.weed_ca.WeedCA.polygon_to_weedca(
        metro,
        pressure_coef=0.000783,
        cell_size=1000,
        growth_rate=0.02,
        diffusion_rate=0.01,
    )

    def proportional_removal(
        weed_grid,
        mask,
        rate=removal_rate,
    ):
        valid_weed = np.where(mask, weed_grid, 0.0)

        diff = valid_weed * rate
        cost = np.sum(diff * 200)

        return diff, cost

    weedca.register_policy(proportional_removal)

    history, cost_history = weedca.simulate(
        steps=108,
        modes=["growth", "diffusion", "removal"],
    )

    results[removal_rate] = {
        "history": history,
        "cost_history": cost_history,
        "mean_weed": np.nanmean(
            history,
            axis=(1, 2),
        ),
        "total_weed": np.nansum(
            history,
            axis=(1, 2),
        ),
    }

In [ ]:
plt.figure(figsize=(10, 6))

for removal_rate, result in results.items():
    plt.plot(
        result["mean_weed"],
        label=f"{removal_rate:.1%} removal",
    )

plt.xlabel("Simulation Step")
plt.ylabel("Mean Weed Index")
plt.title("Effect of Removal Rate on Weed Growth")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for removal_rate, result in results.items():
    plt.plot(
        result["cost_history"],
        label=f"{removal_rate:.1%} removal",
    )

plt.xlabel("Simulation Step")
plt.ylabel("Cumulative Cost")
plt.title("Cumulative Removal Cost")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
final_mean = results[0.1]["mean_weed"][-1]

print(f"Final mean weed index: {final_mean:.12f}")